In [0]:
base = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "silver/state_elections/"
)

df_nrw_2017 = spark.read.parquet(
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/gold/state_election/state=nrw/election_year=2017/"
)

df_nrw_2022 = spark.read.parquet(
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/gold/state_election/state=nrw/election_year=2022/"
)

df_hamburg_2020 = spark.read.parquet(
    base + "state=hamburg/election_year=2020/"
)

df_hamburg_2025 = spark.read.parquet(
    base + "state=hamburg/election_year=2025/"
)

df_sachsen_2019 = spark.read.parquet(
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/gold/state_election/state=sachsen/election_year=2019/"
)

df_sachsen_2024 = spark.read.parquet(
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/gold/state_election/state=sachsen/election_year=2024/"
)

df_sa_2021 = spark.read.parquet(
    base + "state=sachsen_anhalt/election_year=2021/"
)

df_sa_2026 = spark.read.parquet(
    base + "state=sachsen_anhalt/election_year=2026/"
)

df_bayern = spark.read.parquet(
    base + "state=bayern/"
)

df_mv = spark.read.parquet(
    base + "state=mecklenburg_vorpommern/election_year=2021/mv2021.csv"
)

In [0]:
print(df_nrw_2022.columns)

['wahljahr', 'wahltyp', 'bundesland', 'wahlkreis_id', 'wahlkreis_name', 'gemeinde_id', 'gemeinde_name', 'wahlbezirk_id', 'wahlbezirk_name', 'gebietstyp', 'wahlberechtigte', 'waehler', 'gueltige_erststimmen', 'ungueltige_erststimmen', 'gueltige_zweitstimmen', 'ungueltige_zweitstimmen']


In [0]:
from pyspark.sql import functions as F
turnout_sa_2026 = (
    df_sa_2026
    .select(
        F.col("wahljahr"),
        F.col("wahltyp"),
        F.col("bundesland"),

        F.col("wahlkreisnummer").cast("string").alias("wahlkreis_id"),
        F.col("wahlkreisname").alias("wahlkreis_name"),

        F.col("gemeindeschluessel").cast("string").alias("gemeinde_id"),
        F.col("gemeindename").alias("gemeinde_name"),

        F.col("wahlbezirk").cast("string").alias("wahlbezirk_id"),
        F.col("wahlbezirksname").alias("wahlbezirk_name"),

        F.lit("Wahlbezirk").alias("gebietstyp"),

        F.col("a_wahlberechtigte").cast("long").alias("wahlberechtigte"),
        F.col("b_waehler").cast("long").alias("waehler"),

        F.col("d_gueltige_erststimmen").cast("long").alias("gueltige_erststimmen"),
        F.col("c_ungueltige_erststimmen").cast("long").alias("ungueltige_erststimmen"),

        F.col("f_gueltige_zweitstimmen").cast("long").alias("gueltige_zweitstimmen"),
        F.col("e_ungueltige_zweitstimmen").cast("long").alias("ungueltige_zweitstimmen")
    )
    .distinct()
)

In [0]:
turnout_sa_2026.printSchema()

print("Rows:", turnout_sa_2026.count())

turnout_sa_2026.show(20, truncate=False)

root
 |-- wahljahr: integer (nullable = true)
 |-- wahltyp: string (nullable = true)
 |-- bundesland: string (nullable = true)
 |-- wahlkreis_id: string (nullable = true)
 |-- wahlkreis_name: string (nullable = true)
 |-- gemeinde_id: string (nullable = true)
 |-- gemeinde_name: string (nullable = true)
 |-- wahlbezirk_id: string (nullable = true)
 |-- wahlbezirk_name: string (nullable = true)
 |-- gebietstyp: string (nullable = false)
 |-- wahlberechtigte: long (nullable = true)
 |-- waehler: long (nullable = true)
 |-- gueltige_erststimmen: long (nullable = true)
 |-- ungueltige_erststimmen: long (nullable = true)
 |-- gueltige_zweitstimmen: long (nullable = true)
 |-- ungueltige_zweitstimmen: long (nullable = true)

Rows: 2661
+--------+------------+--------------+------------+-------------------+-----------+-------------------------------+-------------+-------------------------------+----------+---------------+-------+--------------------+----------------------+--------------------

In [0]:
turnout_sa_2026.select(
    "wahlberechtigte",
    "waehler",
    "gueltige_erststimmen",
    "ungueltige_erststimmen",
    "gueltige_zweitstimmen",
    "ungueltige_zweitstimmen"
).show(20)

+---------------+-------+--------------------+----------------------+---------------------+-----------------------+
|wahlberechtigte|waehler|gueltige_erststimmen|ungueltige_erststimmen|gueltige_zweitstimmen|ungueltige_zweitstimmen|
+---------------+-------+--------------------+----------------------+---------------------+-----------------------+
|            421|    248|                 244|                     4|                  246|                      2|
|            144|     97|                  95|                     2|                   94|                      3|
|            510|    317|                 316|                     1|                  316|                      1|
|           1589|    764|                 752|                    12|                  750|                     14|
|            246|    150|                 149|                     1|                  149|                      1|
|            292|    203|                 198|                     5|   

In [0]:
turnout_sa_2021 = (
    df_sa_2021
    .select(
        F.col("wahljahr"),
        F.col("wahltyp"),
        F.col("bundesland"),

        F.col("wahlkreisnr").cast("string").alias("wahlkreis_id"),
        F.col("wahlkreisname").alias("wahlkreis_name"),

        F.col("gemeindeschluessel").cast("string").alias("gemeinde_id"),
        F.col("gemeindename").alias("gemeinde_name"),

        F.col("wahlbezirk_nr").cast("string").alias("wahlbezirk_id"),
        F.col("wahlbezirk_name").alias("wahlbezirk_name"),

        F.col("wahllokalart").alias("gebietstyp"),

        F.col("wahlberechtigte_insgesamt").cast("long").alias("wahlberechtigte"),
        F.col("waehler_insgesamt").cast("long").alias("waehler"),

        F.col("gueltige_erststimmen").cast("long").alias("gueltige_erststimmen"),
        F.col("ungueltige_erststimmen").cast("long").alias("ungueltige_erststimmen"),

        F.col("gueltige_zweitstimmen").cast("long").alias("gueltige_zweitstimmen"),
        F.col("ungueltige_zweitstimmen").cast("long").alias("ungueltige_zweitstimmen")
    )
    .distinct()
)

In [0]:
turnout_sa_2021.printSchema()

print("Rows:", turnout_sa_2021.count())

turnout_sa_2021.show(20, truncate=False)

root
 |-- wahljahr: integer (nullable = true)
 |-- wahltyp: string (nullable = true)
 |-- bundesland: string (nullable = true)
 |-- wahlkreis_id: string (nullable = true)
 |-- wahlkreis_name: string (nullable = true)
 |-- gemeinde_id: string (nullable = true)
 |-- gemeinde_name: string (nullable = true)
 |-- wahlbezirk_id: string (nullable = true)
 |-- wahlbezirk_name: string (nullable = true)
 |-- gebietstyp: string (nullable = true)
 |-- wahlberechtigte: long (nullable = true)
 |-- waehler: long (nullable = true)
 |-- gueltige_erststimmen: long (nullable = true)
 |-- ungueltige_erststimmen: long (nullable = true)
 |-- gueltige_zweitstimmen: long (nullable = true)
 |-- ungueltige_zweitstimmen: long (nullable = true)

Rows: 2628
+--------+------------+--------------+------------+--------------------+-----------+-------------------------------+-------------+----------------------------------+----------+---------------+-------+--------------------+----------------------+-----------------

In [0]:
turnout_sa_2021 = (
    turnout_sa_2021
    .withColumn(
        "check_erst",
        F.col("waehler")
        - (
            F.col("gueltige_erststimmen")
            + F.col("ungueltige_erststimmen")
        )
    )
    .withColumn(
        "check_zweit",
        F.col("waehler")
        - (
            F.col("gueltige_zweitstimmen")
            + F.col("ungueltige_zweitstimmen")
        )
    )
)

In [0]:
turnout_sa_2021.groupBy(
    "check_erst",
    "check_zweit"
).count().orderBy(
    F.desc("count")
).show(30)

+----------+-----------+-----+
|check_erst|check_zweit|count|
+----------+-----------+-----+
|         0|          0| 2397|
|         0|       NULL|   97|
|      NULL|       NULL|   96|
|      NULL|          0|   38|
+----------+-----------+-----+



In [0]:
turnout_sa_2021 = turnout_sa_2021.drop(
    "check_erst",
    "check_zweit"
)

turnout_sa_2026 = turnout_sa_2026.drop(
    "check_erst",
    "check_zweit"
)

In [0]:
turnout_sachsenanhalt = (
    turnout_sa_2021
    .unionByName(turnout_sa_2026)
)

In [0]:
print("Sachsen 2024")
print(df_sachsen_2024.columns)

print("Sachsen 2019")
print(df_sachsen_2019.columns)

Sachsen 2024
['wahljahr', 'wahltyp', 'bundesland', 'wahlkreis_id', 'wahlkreis_name', 'gemeinde_id', 'gemeinde_name', 'wahlbezirk_id', 'wahlbezirk_name', 'gebietstyp', 'wahlberechtigte', 'waehler', 'gueltige_erststimmen', 'ungueltige_erststimmen', 'gueltige_zweitstimmen', 'ungueltige_zweitstimmen']
Sachsen 2019
['wahljahr', 'wahltyp', 'bundesland', 'wahlkreis_id', 'wahlkreis_name', 'gemeinde_id', 'gemeinde_name', 'wahlbezirk_id', 'wahlbezirk_name', 'gebietstyp', 'wahlberechtigte', 'waehler', 'gueltige_erststimmen', 'ungueltige_erststimmen', 'gueltige_zweitstimmen', 'ungueltige_zweitstimmen', 'check_erst', 'check_zweit']


In [0]:
df_sachsen_2019 =df_sachsen_2019.drop(
    "check_erst",
    "check_zweit"
)

In [0]:
turnout_sachsen_2019=df_sachsen_2019
turnout_sachsen_2024=df_sachsen_2024
print(turnout_sachsen_2019.columns)
print(turnout_sachsen_2024.columns)

['wahljahr', 'wahltyp', 'bundesland', 'wahlkreis_id', 'wahlkreis_name', 'gemeinde_id', 'gemeinde_name', 'wahlbezirk_id', 'wahlbezirk_name', 'gebietstyp', 'wahlberechtigte', 'waehler', 'gueltige_erststimmen', 'ungueltige_erststimmen', 'gueltige_zweitstimmen', 'ungueltige_zweitstimmen']
['wahljahr', 'wahltyp', 'bundesland', 'wahlkreis_id', 'wahlkreis_name', 'gemeinde_id', 'gemeinde_name', 'wahlbezirk_id', 'wahlbezirk_name', 'gebietstyp', 'wahlberechtigte', 'waehler', 'gueltige_erststimmen', 'ungueltige_erststimmen', 'gueltige_zweitstimmen', 'ungueltige_zweitstimmen']


In [0]:
turnout_sachsen = (
    turnout_sachsen_2019
    .unionByName(turnout_sachsen_2024)
)

In [0]:
print(df_hamburg_2020.columns)

print(df_hamburg_2025.columns)

['bezirk', 'wahlkreis', 'wahlbezirk', 'wahlberechtigte_ohne_wahlschein', 'wahlberechtigte_mit_wahlschein', 'wahlberechtigte', 'waehlende', 'briefwaehlende', 'abgegebene_stimmzettel', 'ungueltige_stimmzettel', 'gueltige_stimmzettel', 'gueltige_stimmen_gesamt', 'gueltige_stimmen_liste', 'gueltige_stimmen_person', 'gueltige_stimmen_hr', 'partei', 'stimmenart', 'stimmen', 'wahljahr', 'wahltyp', 'bundesland']
['bezirk', 'wahlkreis', 'stadtteil', 'erfassungsgebietsnummer', 'erfassungsgebietsart', 'wahlberechtigte_gesamt_a', 'wahlberechtigte_ohne_wahlschein_a1', 'wahlberechtigte_mit_wahlschein_a2', 'wahlberechtigte_nicht_im_wvz_a3', 'waehler_gesamt_b', 'waehler_mit_wahlschein_b1', 'waehler_ohne_wahlschein_b2', 'stimmzettel_gesamt_b4', 'stimmzettel_ungueltig_e1', 'stimmzettel_gueltig_e2', 'stimmen_gueltige_f', 'partei', 'stimmenart', 'stimmen', 'wahljahr', 'wahltyp', 'bundesland']


In [0]:
turnout_hamburg_2020 = (
    df_hamburg_2020
    .select(
        "wahljahr",
        "wahltyp",
        "bundesland",

        F.lit(None).cast("string").alias("wahlkreis_id"),
        F.col("wahlkreis").alias("wahlkreis_name"),

        F.lit(None).cast("string").alias("gemeinde_id"),
        F.col("bezirk").alias("gemeinde_name"),

        F.col("wahlbezirk").cast("string").alias("wahlbezirk_id"),
        F.lit(None).cast("string").alias("wahlbezirk_name"),

        F.lit("Wahlbezirk").alias("gebietstyp"),

        F.col("wahlberechtigte"),
        F.col("waehlende").alias("waehler"),

        F.lit(None).cast("long").alias("gueltige_erststimmen"),
        F.lit(None).cast("long").alias("ungueltige_erststimmen"),
        F.lit(None).cast("long").alias("gueltige_zweitstimmen"),
        F.lit(None).cast("long").alias("ungueltige_zweitstimmen")
    )
    .distinct()
)

In [0]:
turnout_hamburg_2025 = (
    df_hamburg_2025
    .select(
        "wahljahr",
        "wahltyp",
        "bundesland",

        F.lit(None).cast("string").alias("wahlkreis_id"),
        F.col("wahlkreis").alias("wahlkreis_name"),

        F.lit(None).cast("string").alias("gemeinde_id"),
        F.col("stadtteil").alias("gemeinde_name"),

        F.col("erfassungsgebietsnummer").cast("string").alias("wahlbezirk_id"),
        F.lit(None).cast("string").alias("wahlbezirk_name"),

        F.col("erfassungsgebietsart").alias("gebietstyp"),

        F.col("wahlberechtigte_gesamt_a").alias("wahlberechtigte"),
        F.col("waehler_gesamt_b").alias("waehler"),

        F.lit(None).cast("long").alias("gueltige_erststimmen"),
        F.lit(None).cast("long").alias("ungueltige_erststimmen"),
        F.lit(None).cast("long").alias("gueltige_zweitstimmen"),
        F.lit(None).cast("long").alias("ungueltige_zweitstimmen")
    )
    .distinct()
)

In [0]:
turnout_hamburg = (
    turnout_hamburg_2020
    .unionByName(turnout_hamburg_2025)
)

In [0]:
turnout_hamburg.groupBy(
    "bundesland",
    "wahljahr"
).count().show()

+----------+--------+-----+
|bundesland|wahljahr|count|
+----------+--------+-----+
|   Hamburg|    2020| 1884|
|   Hamburg|    2025| 1972|
+----------+--------+-----+



In [0]:
turnout_bayern = (
    df_bayern
    .select(
        "wahljahr",
        "wahltyp",
        "bundesland",

        F.lit(None).cast("string").alias("wahlkreis_id"),
        F.lit(None).cast("string").alias("wahlkreis_name"),

        F.lit(None).cast("string").alias("gemeinde_id"),
        F.col("stadt").alias("gemeinde_name"),

        F.col("gebietsnummer").cast("string").alias("wahlbezirk_id"),
        F.lit(None).cast("string").alias("wahlbezirk_name"),

        F.col("stimmbezirksart").alias("gebietstyp"),

        F.col("wahlberechtigte"),
        F.col("waehler"),

        F.lit(None).cast("long").alias("gueltige_erststimmen"),
        F.lit(None).cast("long").alias("ungueltige_erststimmen"),
        F.lit(None).cast("long").alias("gueltige_zweitstimmen"),
        F.lit(None).cast("long").alias("ungueltige_zweitstimmen")
    )
    .distinct()
)

In [0]:
turnout_mv = (
    df_mv
    .groupBy(
        "Wahljahr",
        "Wahltyp",
        "Bundesland",
        "Wahlkreis",
        "Wahlkreisname",
        "Gemeinde",
        "Gemeindename",
        "Wahlbezirk",
        "Wahlbezirksname",
        "Wahlberechtigte",
        "Wähler"
    )
    .agg(
        F.max(
            F.when(
                F.col("Erst-/Zweitstimme") == "1",
                F.col("Gueltige_Stimmen_Anzahl")
            )
        ).alias("gueltige_erststimmen"),

        F.max(
            F.when(
                F.col("Erst-/Zweitstimme") == "1",
                F.col("Ungueltige_Stimmen_Anzahl")
            )
        ).alias("ungueltige_erststimmen"),

        F.max(
            F.when(
                F.col("Erst-/Zweitstimme") == "2",
                F.col("Gueltige_Stimmen_Anzahl")
            )
        ).alias("gueltige_zweitstimmen"),

        F.max(
            F.when(
                F.col("Erst-/Zweitstimme") == "2",
                F.col("Ungueltige_Stimmen_Anzahl")
            )
        ).alias("ungueltige_zweitstimmen")
    )
    .select(
        F.col("Wahljahr").alias("wahljahr"),
        F.col("Wahltyp").alias("wahltyp"),
        F.col("Bundesland").alias("bundesland"),

        F.col("Wahlkreis").cast("string").alias("wahlkreis_id"),
        F.col("Wahlkreisname").alias("wahlkreis_name"),

        F.col("Gemeinde").cast("string").alias("gemeinde_id"),
        F.col("Gemeindename").alias("gemeinde_name"),

        F.col("Wahlbezirk").cast("string").alias("wahlbezirk_id"),
        F.col("Wahlbezirksname").alias("wahlbezirk_name"),

        F.lit("Wahlbezirk").alias("gebietstyp"),

        F.col("Wahlberechtigte").alias("wahlberechtigte"),
        F.col("Wähler").alias("waehler"),

        "gueltige_erststimmen",
        "ungueltige_erststimmen",
        "gueltige_zweitstimmen",
        "ungueltige_zweitstimmen"
    )
)

In [0]:
turnout_nrw_2017 = df_nrw_2017
turnout_nrw_2022 = df_nrw_2022

In [0]:
print(df_nrw_2022.columns)

['wahljahr', 'wahltyp', 'bundesland', 'wahlkreis_id', 'wahlkreis_name', 'gemeinde_id', 'gemeinde_name', 'wahlbezirk_id', 'wahlbezirk_name', 'gebietstyp', 'wahlberechtigte', 'waehler', 'gueltige_erststimmen', 'ungueltige_erststimmen', 'gueltige_zweitstimmen', 'ungueltige_zweitstimmen']


In [0]:
print(df_nrw_2017.columns)

['wahljahr', 'wahltyp', 'bundesland', 'wahlkreis_id', 'wahlkreis_name', 'gemeinde_id', 'gemeinde_name', 'wahlbezirk_id', 'wahlbezirk_name', 'gebietstyp', 'wahlberechtigte', 'waehler', 'gueltige_erststimmen', 'ungueltige_erststimmen', 'gueltige_zweitstimmen', 'ungueltige_zweitstimmen']


In [0]:
print(df_nrw_2017.columns == df_nrw_2022.columns)

True


In [0]:
turnout_nrw = (
    df_nrw_2017
    .unionByName(df_nrw_2022)
)

turnout_nrw.groupBy(
    "bundesland",
    "wahljahr"
).count().show()

+-------------------+--------+-----+
|         bundesland|wahljahr|count|
+-------------------+--------+-----+
|Nordrhein-Westfalen|    2017|  129|
|Nordrhein-Westfalen|    2022|  129|
+-------------------+--------+-----+



In [0]:
print(turnout_mv.columns==turnout_bayern.columns==turnout_hamburg.columns==turnout_nrw.columns==turnout_sachsen.columns==turnout_sachsenanhalt.columns)

True


In [0]:
gold_state_turnout = (
    turnout_nrw_2017
    .unionByName(turnout_nrw_2022)
    .unionByName(turnout_hamburg_2020)
    .unionByName(turnout_hamburg_2025)
    .unionByName(turnout_sachsen_2019)
    .unionByName(turnout_sachsen_2024)
    .unionByName(turnout_sa_2021)
    .unionByName(turnout_sa_2026)
    .unionByName(turnout_bayern)
    .unionByName(turnout_mv)
)

In [0]:
gold_state_turnout.printSchema()

print("Rows:", gold_state_turnout.count())

gold_state_turnout.groupBy(
    "bundesland",
    "wahljahr"
).count().orderBy(
    "bundesland",
    "wahljahr"
).show()

root
 |-- wahljahr: integer (nullable = true)
 |-- wahltyp: string (nullable = true)
 |-- bundesland: string (nullable = true)
 |-- wahlkreis_id: string (nullable = true)
 |-- wahlkreis_name: string (nullable = true)
 |-- gemeinde_id: string (nullable = true)
 |-- gemeinde_name: string (nullable = true)
 |-- wahlbezirk_id: string (nullable = true)
 |-- wahlbezirk_name: string (nullable = true)
 |-- gebietstyp: string (nullable = true)
 |-- wahlberechtigte: long (nullable = true)
 |-- waehler: long (nullable = true)
 |-- gueltige_erststimmen: long (nullable = true)
 |-- ungueltige_erststimmen: long (nullable = true)
 |-- gueltige_zweitstimmen: long (nullable = true)
 |-- ungueltige_zweitstimmen: long (nullable = true)

Rows: 21292
+--------------------+--------+-----+
|          bundesland|wahljahr|count|
+--------------------+--------+-----+
|              Bayern|    2023| 1028|
|             Hamburg|    2020| 1884|
|             Hamburg|    2025| 1972|
|Mecklenburg-Vorpo...|    2021| 

In [0]:
print(
    "Exact duplicates:",
    gold_state_turnout.count()
    - gold_state_turnout.distinct().count()
)

Exact duplicates: 0


In [0]:
gold_state_turnout.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in gold_state_turnout.columns
]).show(truncate=False)

+--------+-------+----------+------------+--------------+-----------+-------------+-------------+---------------+----------+---------------+-------+--------------------+----------------------+---------------------+-----------------------+
|wahljahr|wahltyp|bundesland|wahlkreis_id|wahlkreis_name|gemeinde_id|gemeinde_name|wahlbezirk_id|wahlbezirk_name|gebietstyp|wahlberechtigte|waehler|gueltige_erststimmen|ungueltige_erststimmen|gueltige_zweitstimmen|ungueltige_zweitstimmen|
+--------+-------+----------+------------+--------------+-----------+-------------+-------------+---------------+----------+---------------+-------+--------------------+----------------------+---------------------+-----------------------+
|0       |0      |0         |4884        |1028          |5142       |258          |258          |5142           |0         |2268           |0      |4884                |5018                  |4884                 |5077                   |
+--------+-------+----------+------------+--

In [0]:
gold_turnout_path = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "gold/election_turnout/state/"
)

(
    gold_state_turnout
    .write
    .mode("overwrite")
    .partitionBy("bundesland", "wahljahr")
    .parquet(gold_turnout_path)
)

In [0]:
turnout_bayern.groupBy(
    "gebietstyp"
).agg(
    F.count("*").alias("rows"),
    F.sum("wahlberechtigte").alias("wahlberechtigte"),
    F.sum("waehler").alias("waehler")
).show()

+----------+----+---------------+-------+
|gebietstyp|rows|wahlberechtigte|waehler|
+----------+----+---------------+-------+
|        11| 507|        1820168| 577794|
|        21| 521|              0| 680598|
+----------+----+---------------+-------+



In [0]:
turnout_bayern.orderBy(
    F.desc("wahlberechtigte")
).select(
    "wahlbezirk_id",
    "gebietstyp",
    "wahlberechtigte",
    "waehler"
).show(30, truncate=False)

+-------------+----------+---------------+-------+
|wahlbezirk_id|gebietstyp|wahlberechtigte|waehler|
+-------------+----------+---------------+-------+
|162          |11        |910084         |288897 |
|2210         |11        |2090           |537    |
|105          |11        |2038           |704    |
|106          |11        |2010           |582    |
|1310         |11        |2004           |652    |
|2216         |11        |2003           |560    |
|2402         |11        |2003           |585    |
|715          |11        |1990           |671    |
|1801         |11        |1979           |598    |
|107          |11        |1975           |686    |
|1010         |11        |1972           |527    |
|2201         |11        |1970           |702    |
|2009         |11        |1969           |633    |
|1014         |11        |1969           |577    |
|1103         |11        |1968           |411    |
|2211         |11        |1967           |607    |
|2120         |11        |1967 

In [0]:
turnout_bayern.groupBy(
    "wahlbezirk_id"
).agg(
    F.countDistinct("gebietstyp").alias("anzahl_typen"),
    F.count("*").alias("rows")
).filter(
    F.col("anzahl_typen") > 1
).show(50, truncate=False)

+-------------+------------+----+
|wahlbezirk_id|anzahl_typen|rows|
+-------------+------------+----+
|162          |2           |3   |
+-------------+------------+----+



In [0]:
turnout_bayern_clean = (
    df_bayern
    .select(
        "gebietsnummer",
        "stimmbezirksart",
        "wahlberechtigte",
        "waehler",
        "wahljahr",
        "wahltyp",
        "bundesland",
        "stadt"
    )
    .distinct()
)

In [0]:
turnout_bayern_clean.groupBy(
    "stimmbezirksart"
).agg(
    F.count("*").alias("rows"),
    F.sum("wahlberechtigte").alias("wahlberechtigte"),
    F.sum("waehler").alias("waehler")
).show()

+---------------+----+---------------+-------+
|stimmbezirksart|rows|wahlberechtigte|waehler|
+---------------+----+---------------+-------+
|             11| 507|        1820168| 577794|
|             21| 521|              0| 680598|
+---------------+----+---------------+-------+



In [0]:
df_bayern.groupBy(
    "gebietsart_schluessel",
    "stimmbezirksart"
).agg(
    F.countDistinct("gebietsnummer").alias("gebiete"),
    F.sum("wahlberechtigte").alias("wahlberechtigte"),
    F.sum("waehler").alias("waehler")
).orderBy(
    "gebietsart_schluessel",
    "stimmbezirksart"
).show(100, truncate=False)

+---------------------+---------------+-------+---------------+--------+
|gebietsart_schluessel|stimmbezirksart|gebiete|wahlberechtigte|waehler |
+---------------------+---------------+-------+---------------+--------+
|LAST                 |11             |507    |81907560       |26000730|
|LAST                 |21             |520    |0              |30626910|
+---------------------+---------------+-------+---------------+--------+



In [0]:
df_bayern.select(
    "gebietsart_schluessel",
    "gebietsnummer",
    "stimmbezirksart",
    "wahlberechtigte",
    "waehler"
).distinct().orderBy(
    F.desc("wahlberechtigte")
).show(50, truncate=False)

+---------------------+-------------+---------------+---------------+-------+
|gebietsart_schluessel|gebietsnummer|stimmbezirksart|wahlberechtigte|waehler|
+---------------------+-------------+---------------+---------------+-------+
|LAST                 |162          |11             |910084         |288897 |
|LAST                 |2210         |11             |2090           |537    |
|LAST                 |105          |11             |2038           |704    |
|LAST                 |106          |11             |2010           |582    |
|LAST                 |1310         |11             |2004           |652    |
|LAST                 |2216         |11             |2003           |560    |
|LAST                 |2402         |11             |2003           |585    |
|LAST                 |715          |11             |1990           |671    |
|LAST                 |1801         |11             |1979           |598    |
|LAST                 |107          |11             |1975       

In [0]:
df_bayern.filter(
    F.col("gebietsnummer") == "162"
).select(
    "gebietsart_schluessel",
    "gebietsnummer",
    "stimmbezirksart",
    "wahlberechtigte",
    "waehler"
).show(truncate=False)

+---------------------+-------------+---------------+---------------+-------+
|gebietsart_schluessel|gebietsnummer|stimmbezirksart|wahlberechtigte|waehler|
+---------------------+-------------+---------------+---------------+-------+
|LAST                 |162          |11             |910084         |288897 |
|LAST                 |162          |21             |0              |340299 |
|LAST                 |162          |21             |0              |613    |
|LAST                 |162          |11             |910084         |288897 |
|LAST                 |162          |21             |0              |340299 |
|LAST                 |162          |21             |0              |613    |
|LAST                 |162          |11             |910084         |288897 |
|LAST                 |162          |21             |0              |340299 |
|LAST                 |162          |21             |0              |613    |
|LAST                 |162          |11             |910084     

In [0]:
bayern_summary = (
    df_bayern
    .filter(F.col("gebietsnummer") == "162")
    .select(
        F.lit("Bayern").alias("bundesland"),
        F.lit(2023).alias("wahljahr"),

        F.max(
            F.when(
                F.col("stimmbezirksart") == "11",
                F.col("wahlberechtigte")
            )
        ).alias("wahlberechtigte"),

        (
            F.max(
                F.when(
                    (F.col("stimmbezirksart") == "11"),
                    F.col("waehler")
                )
            )
            +
            F.max(
                F.when(
                    (F.col("stimmbezirksart") == "21"),
                    F.col("waehler")
                )
            )
        ).alias("waehler")
    )
)

In [0]:
bayern_summary = bayern_summary.withColumn(
    "wahlbeteiligung_pct",
    F.round(
        F.col("waehler") / F.col("wahlberechtigte") * 100,
        2
    )
)

In [0]:
bayern_summary_path="abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/gold/bayern_summary_path/"

bayern_summary.write.mode("overwrite").parquet(bayern_summary_path)